# Banana Disease Detection System
Train a multi-class image classification model using TensorFlow, Keras, and EfficientNetB0 transfer learning to detect banana diseases.

In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

try:
    from PIL import Image
except ImportError:
    raise ImportError("Pillow is not installed. Please install it using `!pip install pillow` natively.")

### 1. Configuration & Data Preprocessing
Define paths, image sizes, and initialize the `ImageDataGenerator` for training (with augmentations) and validation (80/20 split).

In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'dataset'
MODEL_DIR = 'model'

CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'banana_model.keras')
LABELS_PATH = os.path.join(MODEL_DIR, 'banana_labels.json')

# Ensure model output directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

# Data Augmentation + 80/20 val split
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

print("Loading Training Data...")
train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

print("Loading Validation Data...")
val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Loading Training Data...
Found 35200 images belonging to 4 classes.
Loading Validation Data...
Found 8800 images belonging to 4 classes.


### 2. Output Class Labels & Handle Imbalance
Map class indices to class folder names and calculate class weights so the model learns fairly from imbalanced classes.

In [3]:
# Map and save class labels
class_indices = train_gen.class_indices
class_labels = {v: k for k, v in class_indices.items()}

with open(LABELS_PATH, 'w') as f:
    json.dump(class_labels, f, indent=4)
print(f"Saved class labels to {LABELS_PATH}: \n", class_labels)

# Compute class weights
classes = train_gen.classes
class_weight_arr = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(classes), 
    y=classes
)

class_weights = {i: weight for i, weight in enumerate(class_weight_arr)}
print(f"\nComputed Class Weights: {class_weights}")

Saved class labels to model\banana_labels.json: 
 {0: 'fusarium_wilt', 1: 'healthy', 2: 'natural_death_leaf', 3: 'rhizome_root'}

Computed Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0)}


### 3. Setup Transfer Learning Model (EfficientNetB0)
Start with base `imagenet` weights frozen, replacing the top layer to match our 4 specific banana classes.

In [4]:
num_classes = len(class_indices)

# Load base model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
base_model.trainable = False # Freeze base initially

# Construct classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4)                   │           5,124 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

### 4. Phase 1: Train Top Layers
Train for the first 20 epochs using early stopping and checkpointing.

In [ ]:
callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath=CHECKPOINT_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

INITIAL_EPOCHS = 20

history = model.fit(
    train_gen,
    epochs=INITIAL_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_list
)

Epoch 1/20
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2675 - loss: 1.3986

### 5. Phase 2: Fine-Tuning the Base Model
Unfreeze the base model weights, lower the learning rate significantly, and train for 10 more epochs to improve accuracy.

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

FINE_TUNE_EPOCHS = 10
TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

history_fine = model.fit(
    train_gen,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history.epoch[-1],
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_list
)

### 6. Predict Function (Ready for Deployment)
This function easily translates to a FastAPI endpoint.

In [ ]:
def load_labels(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

def predict_disease(img_path):
    # Validation checks
    if not os.path.exists(CHECKPOINT_PATH):
        raise ValueError(f"Model not found at {CHECKPOINT_PATH}")
        
    loaded_model = tf.keras.models.load_model(CHECKPOINT_PATH)
    labels_dict = load_labels(LABELS_PATH)
    
    # Preprocess Image
    img = load_img(img_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Training normalization scale
    
    # Inference
    predictions = loaded_model.predict(img_array)[0]
    predicted_class_index = np.argmax(predictions)
    confidence = float(predictions[predicted_class_index])
    
    predicted_class_name = labels_dict[str(predicted_class_index)]
    
    return {
        "disease": predicted_class_name,
        "confidence": round(confidence * 100, 2)
    }

# Example Usage:
# result = predict_disease('dataset/fusarium_wilt/sample_img.jpg')
# print(result) # Output: {'disease': 'fusarium_wilt', 'confidence': 98.4}